In [14]:
%pip install python-dotenv groq langchain-groq --quiet

import os
import getpass
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


if not os.getenv("GROQ_API_KEY"):
    print("GROQ_API_KEY not found in .env file.")
    os.environ["GROQ_API_KEY"] = getpass.getpass("Please enter your Groq API Key: ")

In [15]:
# 2. Updated Expert Configurations (2026 Models)
MODEL_CONFIG = {
    "technical": {
        "system": "You are a rigorous Technical Support Expert. You provide precise, code-focused solutions. Always check for common syntax errors and explain the logic behind your fix.",
        "model": "llama-3.3-70b-versatile", # Updated from mixtral
        "temperature": 0.4
    },
    "billing": {
        "system": "You are an empathetic Billing Expert. You focus on financial inquiries and policy-driven solutions. Be helpful and clear about refund timelines and subscription terms.",
        "model": "llama-3.3-70b-versatile", # Updated from mixtral
        "temperature": 0.3
    },
    "general": {
        "system": "You are a friendly General Support Assistant. You handle casual inquiries and general questions with a helpful and polite tone.",
        "model": "llama-3.1-8b-instant", # Using a smaller/faster model for general chat
        "temperature": 0.7
    }
}

print(" Expert configurations updated to Llama-3.3 and Llama-3.1.")

 Expert configurations updated to Llama-3.3 and Llama-3.1.


In [16]:
# The Router (The Core Task)
def route_prompt(user_input):
    # Requirement: Use temperature=0 for consistency in classification
    router_llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.0)

    # Requirement: Return ONLY the category name
    router_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a routing specialist. Classify the user query into one of these categories: 'technical', 'billing', or 'general'. Return ONLY the category name as a single word."),
        ("human", "{query}")
    ])

    # LCEL Chain for routing
    router_chain = router_prompt | router_llm | StrOutputParser()

    # Invoke and clean the output
    category = router_chain.invoke({"query": user_input}).strip().lower()
    return category

# Quick test to see if it works!
test_query = "How do I fix a NullPointerException in my Java code?"
print(f"Query: {test_query}")
print(f"Routed to: {route_prompt(test_query)}")

Query: How do I fix a NullPointerException in my Java code?
Routed to: technical


In [17]:
# 4. The Orchestrator
def process_request(user_input):
    # 1. Calls route_prompt to decide the category
    category = route_prompt(user_input)

    # Validation: Fallback to general if the router returns something unexpected
    if category not in MODEL_CONFIG:
        category = "general"

    print(f"--- Routing to: {category.upper()} Expert ---")

    # 2. Selects the correct System Prompt and config based on the category
    config = MODEL_CONFIG[category]

    # 3. Calls the generic LLM (Mixtral) with that specific System Prompt + User Input
    expert_llm = ChatGroq(
        model=config["model"],
        temperature=config["temperature"]
    )

    expert_prompt = ChatPromptTemplate.from_messages([
        ("system", config["system"]),
        ("human", "{query}")
    ])

    # 4. Returns the final answer
    final_chain = expert_prompt | expert_llm | StrOutputParser()
    return final_chain.invoke({"query": user_input})

# --- RUNNING THE EXAMPLES ---

print("EXAMPLE 1:")
print(process_request("My python script is throwing an IndexError on line 5."))

print("\n" + "="*50 + "\n")

print("EXAMPLE 2:")
print(process_request("I was charged twice for my subscription this month."))

EXAMPLE 1:
--- Routing to: TECHNICAL Expert ---
To assist you with the `IndexError` on line 5 of your Python script, I'll need more information about the code. However, I can guide you through a general troubleshooting process.

### Common Causes of `IndexError`

1. **Out-of-range indexing**: When you try to access an element in a list or other sequence that doesn't exist (e.g., trying to access `my_list[10]` when `my_list` only has 5 elements).
2. **Empty sequences**: Attempting to access any element in an empty list or sequence (e.g., `my_list[0]` when `my_list` is `[]`).

### Steps to Troubleshoot

1. **Check the Line of Code**: Look at line 5 of your script and identify the operation causing the error. It's likely an indexing operation (`my_list[index]` or `my_string[index]`).
2. **Verify Index Values**: Ensure that the index you're trying to access is within the bounds of the sequence (list, string, tuple, etc.).
3. **Sequence Length**: Before accessing an element, you might want 